<a href="https://colab.research.google.com/github/tjdux/Introduction-to-Machine-Learning-with-Python/blob/main/04_1_%EB%B2%94%EC%A3%BC%ED%98%95_%EB%B3%80%EC%88%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 노트북이 코랩에서 실행 중인지 체크합니다.
import os
import sys
if 'google.colab' in sys.modules:
    if not os.path.isdir('mglearn'):
        # mglearn을 다운받고 압축을 풉니다.
        !wget -q -O mglearn.tar.gz https://bit.ly/mglearn-tar-gz
        !tar -xzf mglearn.tar.gz
    !wget -q -O data.tar.gz https://bit.ly/data-tar-gz
    !tar -xzf data.tar.gz
    # 나눔 폰트를 설치합니다.
    !sudo apt-get -qq -y install fonts-nanum
    import matplotlib.font_manager as fm
    font_files = fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum'])
    for fpath in font_files:
        fm.fontManager.addfont(fpath)

In [ ]:

import sklearn
from preamble import *
import matplotlib

# 나눔 폰트를 사용합니다.
matplotlib.rc('font', family='NanumBarunGothic')
matplotlib.rcParams['axes.unicode_minus'] = False

- ⚠️ 지금까지는 입력 데이터가 연속형 특성이었지만, 이번에는 입력 데이터가 범주형 특성인 것! (출력 데이터가 아님을 주의: 출력 데이터에 대해서는 분류 / 회귀로 나눠짐)
## 01 사용할 데이터
- 미국 성인의 소득 데이터 셋
- 소득이 <=50K와 >50K 두 클래스를 가진 분류 문제
- `age`, `hours-per-week`: 연속형 특성
- `workclass`, `education`, `sex`, `occupation`: 범주형 특성
- 로지스틱 회귀 분류기 학습 ⬅️ 로지스틱 회귀 분류를 하기 위해서는 범주형 특성을 다른 방식으로 표현해야 함

## 02 원-핫-인코딩(가변수)
- 가변수: 범주형 변수를 0 또는 1 값을 가진 하나 이상의 새로운 특성으로 바꾼 것
- e.g. `workclass` 특성

|workclass|Government Employee|Private Employee|Self Employed|Self Employed Incorporated|
|---|---|---|---|---|
Government Employee|1|0|0|0|
Private Employee|0|1|0|0|
Self Employed|0|0|1|0|
Self Employed Incorporated|0|0|0|1|



In [ ]:
import os

data = pd.read_csv(
    os.path.join(mglearn.datasets.DATA_PATH, "adult.data"),
    header=None, index_col=False,
    names=["age", "workclass", "fnlwgt", "education", "education-num",
           "martial-status", "occupation", "relationship", "race", "gender",
           "capital-gain", "capital-loss", "hours-per-week", "native-country",
           "income"]
)
data = data[['age', 'workclass', 'education', 'gender', 'hours-per-week',
             'occupation', 'income']]
data.head()

,age,workclass,education,gender,hours-per-week,occupation,income
0,39,State-gov,Bachelors,Male,40,Adm-clerical,<=50K
1,50,Self-emp-not-inc,Bachelors,Male,13,Exec-managerial,<=50K
2,38,Private,HS-grad,Male,40,Handlers-cleaners,<=50K
3,53,Private,11th,Male,40,Handlers-cleaners,<=50K
4,28,Private,Bachelors,Female,40,Prof-specialty,<=50K


### 2.1 문자열로 된 범주형 데이터 확인하기

In [ ]:
# 범주형 데이터 내용 확인
for col in ["workclass", "education", "gender", "occupation", "income"]:
  print(f"{data[col].value_counts()}")
  print()

workclass
Private             22696
Self-emp-not-inc     2541
Local-gov            2093
?                    1836
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64

education
HS-grad         10501
Some-college     7291
Bachelors        5355
Masters          1723
Assoc-voc        1382
11th             1175
Assoc-acdm       1067
10th              933
7th-8th           646
Prof-school       576
9th               514
12th              433
Doctorate         413
5th-6th           333
1st-4th           168
Preschool          51
Name: count, dtype: int64

gender
Male      21790
Female    10771
Name: count, dtype: int64

occupation
Prof-specialty       4140
Craft-repair         4099
Exec-managerial      4066
Adm-clerical         3770
Sales                3650
Other-service        3295
Machine-op-inspct    2002
?                    1843
Transport-moving     1597
Handlers-cleaners    1370
Far

In [ ]:
# pd.get_dummies(): 객체 타입이나 범주형을 가진 열을 자동으로 인코딩

print(f"원본 특성:\n{list(data.columns)}\n")
data_dummies = pd.get_dummies(data)
print(f"get_dummies 후의 특성:\n{list(data_dummies.columns)}")

원본 특성:
['age', 'workclass', 'education', 'gender', 'hours-per-week', 'occupation', 'income']

get_dummies 후의 특성:
['age', 'hours-per-week', 'workclass_ ?', 'workclass_ Federal-gov', 'workclass_ Local-gov', 'workclass_ Never-worked', 'workclass_ Private', 'workclass_ Self-emp-inc', 'workclass_ Self-emp-not-inc', 'workclass_ State-gov', 'workclass_ Without-pay', 'education_ 10th', 'education_ 11th', 'education_ 12th', 'education_ 1st-4th', 'education_ 5th-6th', 'education_ 7th-8th', 'education_ 9th', 'education_ Assoc-acdm', 'education_ Assoc-voc', 'education_ Bachelors', 'education_ Doctorate', 'education_ HS-grad', 'education_ Masters', 'education_ Preschool', 'education_ Prof-school', 'education_ Some-college', 'gender_ Female', 'gender_ Male', 'occupation_ ?', 'occupation_ Adm-clerical', 'occupation_ Armed-Forces', 'occupation_ Craft-repair', 'occupation_ Exec-managerial', 'occupation_ Farming-fishing', 'occupation_ Handlers-cleaners', 'occupation_ Machine-op-inspct', 'occupation_ Oth

In [ ]:
data_dummies.head()

,age,hours-per-week,workclass_ ?,workclass_ Federal-gov,...,occupation_ Tech-support,occupation_ Transport-moving,income_ <=50K,income_ >50K
0,39,40,False,False,...,False,False,True,False
1,50,13,False,False,...,False,False,True,False
2,38,40,False,False,...,False,False,True,False
3,53,40,False,False,...,False,False,True,False
4,28,40,False,False,...,False,False,True,False


- `data_dummies`의 `value` 속성을 이용해 DataFrame을 NumPy 배열로 바꾸어 머신러닝 모델을 학습
- ⚠️ 타깃 값 (`income`) 분리하기❗❗

In [ ]:
# age ~ occupation_ Transport-moving 추출
features = data_dummies.loc[:, "age": "occupation_ Transport-moving"]

# NumPy 배열 추출
X = features.values
y = data_dummies["income_ >50K"].values

print(f"X.shape: {X.shape}  y.shape: {y.shape}")

X.shape: (32561, 44)  y.shape: (32561,)


In [ ]:
# 로지스틱 회귀 분류
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
print(f"테스트 점수: {round(logreg.score(X_test, y_test), 2)}")

테스트 점수: 0.81


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 02 숫자로 표현된 범주형 특성
- ⚠️ 범주형 특성은 종종 숫자로 인코딩되며, 특성의 값이 숫자라고 해서 연속형 특성으로 다뤄야 한다는 의미는 아님
- `get_dummies()`는 숫자 특성은 모두 연속형이라고 생각하여 가변수를 만들지 않음

In [ ]:
# 숫자 특성과 범주형 문자열 특성을 가진 DataFrame
demo_df = pd.DataFrame({"숫자 특성": [0, 1, 2, 1],
                        "범주형 특성": ["양말", "여우", "양말", "상자"]})
demo_df

,숫자 특성,범주형 특성
0,0,양말
1,1,여우
2,2,양말
3,1,상자


In [ ]:
# get_dummies(): 문자열 특성만 인코딩
pd.get_dummies(demo_df)

,숫자 특성,범주형 특성_상자,범주형 특성_양말,범주형 특성_여우
0,0,False,True,False
1,1,False,False,True
2,2,False,True,False
3,1,True,False,False


In [ ]:
# 숫자 특성도 인코딩하고 싶다면 columns 매개변수에 인코딩하고 싶은 열 명시
demo_df["숫자 특성"] = demo_df["숫자 특성"].astype(str)
pd.get_dummies(demo_df, columns=["숫자 특성", "범주형 특성"])

,숫자 특성_0,숫자 특성_1,숫자 특성_2,범주형 특성_상자,범주형 특성_양말,범주형 특성_여우
0,True,False,False,False,True,False
1,False,True,False,False,False,True
2,False,False,True,False,True,False
3,False,True,False,True,False,False
